# 06 - Mushroom modelvergelijking en foutenanalyse

**Auteur:** Brent Van Gelder  
**Dataset:** `mushroom_project_dataset.csv`  
**Teststatus:** gesloten

Deze notebook vergelijkt de belangrijkste modellen onder identieke omstandigheden met repeated stratified cross-validation. Daarna onderzoeken we concrete fouten van het bevroren soft-votingensemble op de bestaande validatieset.

> Dit is een onderwijsmodel en mag nooit worden gebruikt om echte paddenstoelen veilig te verklaren.

## 1. Onderzoeksopzet

We vergelijken Logistic Regression, Histogram Gradient Boosting, Extra Trees, de getunede Random Forest en het bevroren soft-votingensemble.

De 70% trainingsset wordt gebruikt voor 5-fold cross-validation met drie herhalingen. Daardoor krijgen we vijftien gekoppelde scores per model. Iedere fold leert imputatie en encoding uitsluitend uit zijn eigen trainingsdeel.

De eerder afgescheiden validatieset wordt alleen gebruikt voor concrete foutenanalyse. De testset wordt geladen en apart gezet, maar nergens voorspeld.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

## 2. Data laden en exact dezelfde split reconstrueren

Dezelfde seed en tweestapssplit als in alle eerdere notebooks garanderen dezelfde trainings-, validatie- en testindices. De twee expliciete ruiskolommen worden opnieuw verwijderd.

In [ ]:
RANDOM_STATE = 42
DATA_PATH = Path("mushroom_project_dataset.csv")
TARGET = "class"
EDIBLE_LABEL = "e"
POISONOUS_LABEL = "p"

FROZEN_WEIGHTS = {
    "random_forest": 0.70,
    "gradient_boosting": 0.05,
    "extra_trees": 0.25,
}
FROZEN_THRESHOLD = 0.46

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset niet gevonden: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns or df[TARGET].isna().any():
    raise ValueError("De doelkolom ontbreekt of bevat lege waarden.")
if set(df[TARGET].unique()) != {EDIBLE_LABEL, POISONOUS_LABEL}:
    raise ValueError(f"Onverwachte labels: {set(df[TARGET].unique())}")

noise_columns = [column for column in df.columns if column.startswith("jumbled_noise_")]
model_df = df.drop(columns=noise_columns).copy()
X = model_df.drop(columns=TARGET)
y = model_df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rijen": [len(X_train), len(X_validation), len(X_test)],
    "aandeel_giftig": [
        (y_train == POISONOUS_LABEL).mean(),
        (y_validation == POISONOUS_LABEL).mean(),
        (y_test == POISONOUS_LABEL).mean(),
    ],
}, index=["train", "validatie", "test"])
display(split_summary.style.format({"aandeel_giftig": "{:.1%}"}))
print(f"Uitgesloten ruiskolommen: {noise_columns}")
print("De testset is afgescheiden en wordt in deze notebook niet gebruikt.")

## 3. Duplicaat- en overlapcontrole

Na het verwijderen van de ruiskolommen kunnen oorspronkelijk verschillende rijen identiek worden. We controleren hoeveel predictorcombinaties dubbel voorkomen en of identieke predictors zowel in ontwikkeling als test terechtkwamen. Dit verandert de bevroren split niet, maar maakt een mogelijk optimistische evaluatie zichtbaar.

In [ ]:
def row_hashes(frame):
    normalized = frame.copy()
    object_columns = normalized.select_dtypes(exclude=np.number).columns
    normalized[object_columns] = normalized[object_columns].fillna("__missing__")
    numeric_columns = normalized.select_dtypes(include=np.number).columns
    normalized[numeric_columns] = normalized[numeric_columns].fillna(-999999.0)
    return pd.util.hash_pandas_object(normalized, index=False)

train_hashes = row_hashes(X_train)
validation_hashes = row_hashes(X_validation)
test_hashes = row_hashes(X_test)

overlap_summary = pd.Series({
    "dubbele_predictorrijen_volledige_dataset": int(row_hashes(X).duplicated(keep=False).sum()),
    "train_validatie_overlap": int(validation_hashes.isin(set(train_hashes)).sum()),
    "train_test_overlap": int(test_hashes.isin(set(train_hashes)).sum()),
    "validatie_test_overlap": int(test_hashes.isin(set(validation_hashes)).sum()),
})
display(overlap_summary.to_frame("aantal"))

## 4. Identieke preprocessing en modeldefinities

Iedere modelpipeline gebruikt dezelfde mediaan-imputatie, missingness-indicatoren en categorische missingwaarde. Logistic Regression krijgt als enige standaardisatie omdat dit algoritme schaalgevoelig is.

De Random Forest en soft-votinggewichten zijn exact de eerder bevroren configuraties.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

def make_preprocessor(scale_numeric=False):
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ], verbose_feature_names_out=False)

def make_models():
    return {
        "logistic_regression": Pipeline([
            ("preprocessor", make_preprocessor(scale_numeric=True)),
            ("classifier", LogisticRegression(
                class_weight="balanced",
                max_iter=3000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            )),
        ]),
        "gradient_boosting": Pipeline([
            ("preprocessor", make_preprocessor()),
            ("classifier", HistGradientBoostingClassifier(
                learning_rate=0.08,
                max_iter=300,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=0.1,
                class_weight="balanced",
                early_stopping=True,
                validation_fraction=0.15,
                random_state=RANDOM_STATE,
            )),
        ]),
        "extra_trees": Pipeline([
            ("preprocessor", make_preprocessor()),
            ("classifier", ExtraTreesClassifier(
                n_estimators=500,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "random_forest": Pipeline([
            ("preprocessor", make_preprocessor()),
            ("classifier", RandomForestClassifier(
                n_estimators=500,
                max_depth=18,
                min_samples_split=10,
                min_samples_leaf=4,
                max_features="sqrt",
                max_samples=0.9,
                class_weight={EDIBLE_LABEL: 1.0, POISONOUS_LABEL: 3.0},
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
    }

## 5. Repeated stratified cross-validation

Elke fold levert kansen en metrics voor alle modellen. Het soft-votingensemble combineert de kansen van Random Forest, Gradient Boosting en Extra Trees met de bevroren gewichten en drempel 0,46.

De scores uit herhaalde folds zijn niet volledig onafhankelijk. De percentielen hieronder zijn daarom empirische stabiliteitsintervallen, geen formele garanties.

In [ ]:
def metrics_from_probability(y_true, poisonous_probability, threshold):
    predictions = np.where(
        poisonous_probability >= threshold, POISONOUS_LABEL, EDIBLE_LABEL
    )
    y_binary = (np.asarray(y_true) == POISONOUS_LABEL).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
    ).ravel()
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "precision_giftig": precision_score(
            y_true, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "recall_giftig": recall_score(
            y_true, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f1_giftig": f1_score(
            y_true, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f2_giftig": fbeta_score(
            y_true,
            predictions,
            beta=2,
            pos_label=POISONOUS_LABEL,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(y_binary, poisonous_probability),
        "average_precision": average_precision_score(y_binary, poisonous_probability),
        "false_negatives": int(fn),
        "false_positives": int(fp),
    }

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE,
)
cv_rows = []
start_time = time.perf_counter()

for fold_number, (fit_positions, score_positions) in enumerate(
    cv.split(X_train, y_train), start=1
):
    X_fit = X_train.iloc[fit_positions]
    y_fit = y_train.iloc[fit_positions]
    X_score = X_train.iloc[score_positions]
    y_score = y_train.iloc[score_positions]

    models = make_models()
    fold_probabilities = {}

    for model_name, model in models.items():
        model.fit(X_fit, y_fit)
        poisonous_index = list(model.classes_).index(POISONOUS_LABEL)
        probability = model.predict_proba(X_score)[:, poisonous_index]
        fold_probabilities[model_name] = probability

        row = metrics_from_probability(y_score, probability, threshold=0.50)
        row.update({"fold": fold_number, "model": model_name})
        cv_rows.append(row)

    ensemble_probability = sum(
        FROZEN_WEIGHTS[name] * fold_probabilities[name]
        for name in FROZEN_WEIGHTS
    )
    ensemble_row = metrics_from_probability(
        y_score,
        ensemble_probability,
        threshold=FROZEN_THRESHOLD,
    )
    ensemble_row.update({"fold": fold_number, "model": "frozen_soft_voting"})
    cv_rows.append(ensemble_row)

    print(f"Fold {fold_number:02d}/15 voltooid")

cv_results = pd.DataFrame(cv_rows)
elapsed_seconds = time.perf_counter() - start_time
print(f"Repeated cross-validation voltooid in {elapsed_seconds:.1f} seconden.")

## 6. Gemiddelden en empirische intervallen

We tonen per model het gemiddelde, de standaardafwijking en het 2,5e–97,5e percentiel over de vijftien folds. Voor het bevroren ensemble zijn vooral accuracy, recall voor giftig en F2 relevant.

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_giftig",
    "recall_giftig",
    "f1_giftig",
    "f2_giftig",
    "roc_auc",
    "average_precision",
]

summary_rows = []
for model_name, group in cv_results.groupby("model"):
    for metric in metric_columns:
        values = group[metric]
        summary_rows.append({
            "model": model_name,
            "metric": metric,
            "mean": values.mean(),
            "std": values.std(ddof=1),
            "p2_5": values.quantile(0.025),
            "p97_5": values.quantile(0.975),
        })

cv_summary = pd.DataFrame(summary_rows)
mean_table = cv_summary.pivot(index="model", columns="metric", values="mean")
mean_table = mean_table.sort_values(["f2_giftig", "accuracy"], ascending=False)
display(mean_table.style.format("{:.3f}"))
print(mean_table.round(3).to_string())

ensemble_intervals = cv_summary.query(
    "model == 'frozen_soft_voting'"
).set_index("metric")
display(ensemble_intervals.style.format("{:.3f}"))
print(ensemble_intervals.round(3).to_string())

plot_data = cv_results.melt(
    id_vars=["fold", "model"],
    value_vars=["accuracy", "recall_giftig", "f2_giftig"],
    var_name="metric",
    value_name="score",
)
g = sns.catplot(
    data=plot_data,
    x="model",
    y="score",
    col="metric",
    kind="box",
    sharey=True,
    height=4,
    aspect=1.1,
)
for axis in g.axes.flat:
    axis.tick_params(axis="x", rotation=45)
g.fig.subplots_adjust(top=0.82)
g.fig.suptitle("Spreiding over 15 gekoppelde folds")
plt.show()

## 7. Gekoppelde verschillen en Wilcoxon-toets

Omdat ieder model op exact dezelfde folds is gemeten, vergelijken we het bevroren ensemble fold voor fold met Random Forest en Gradient Boosting. De Wilcoxon signed-rank-toets onderzoekt of de mediaan van het gekoppelde verschil nul is.

Met slechts vijftien afhankelijke folds moeten p-waarden voorzichtig worden geïnterpreteerd. Effectgrootte en consistentie zijn minstens zo belangrijk.

In [ ]:
def paired_comparison(metric, reference_model):
    wide = cv_results.pivot(index="fold", columns="model", values=metric)
    differences = wide["frozen_soft_voting"] - wide[reference_model]
    try:
        statistic, p_value = wilcoxon(differences)
    except ValueError:
        statistic, p_value = np.nan, np.nan
    return {
        "metric": metric,
        "vergelijking": f"ensemble - {reference_model}",
        "gemiddeld_verschil": differences.mean(),
        "mediaan_verschil": differences.median(),
        "ensemble_beter_folds": int((differences > 0).sum()),
        "gelijk_folds": int((differences == 0).sum()),
        "ensemble_slechter_folds": int((differences < 0).sum()),
        "wilcoxon_statistic": statistic,
        "p_value_ongecorrigeerd": p_value,
    }

paired_rows = []
for metric in ["accuracy", "recall_giftig", "f2_giftig"]:
    for reference in ["random_forest", "gradient_boosting"]:
        paired_rows.append(paired_comparison(metric, reference))

paired_results = pd.DataFrame(paired_rows)
display(paired_results.style.format({
    "gemiddeld_verschil": "{:+.3f}",
    "mediaan_verschil": "{:+.3f}",
    "p_value_ongecorrigeerd": "{:.4f}",
}))
print(paired_results.round(4).to_string(index=False))

## 8. Bevroren ensemble opnieuw fitten voor foutenanalyse

Voor de foutenanalyse trainen we Random Forest, Gradient Boosting en Extra Trees op de volledige oorspronkelijke trainingsset. Daarna passen we de bevroren gewichten en drempel exact één keer toe op de bestaande validatieset.

In [ ]:
analysis_models = make_models()
analysis_probabilities = {}

for model_name in FROZEN_WEIGHTS:
    model = analysis_models[model_name]
    model.fit(X_train, y_train)
    poisonous_index = list(model.classes_).index(POISONOUS_LABEL)
    analysis_probabilities[model_name] = model.predict_proba(
        X_validation
    )[:, poisonous_index]

validation_probability = sum(
    FROZEN_WEIGHTS[name] * analysis_probabilities[name]
    for name in FROZEN_WEIGHTS
)
validation_prediction = np.where(
    validation_probability >= FROZEN_THRESHOLD,
    POISONOUS_LABEL,
    EDIBLE_LABEL,
)

validation_metrics = pd.Series(
    metrics_from_probability(
        y_validation,
        validation_probability,
        threshold=FROZEN_THRESHOLD,
    ),
    name="bevroren_ensemble_validatie",
)
display(validation_metrics.to_frame().T.style.format("{:.3f}"))
print(validation_metrics.round(3).to_string())

## 9. Welke fouten maakt het ensemble?

Een false negative is hier het ernstigste geval: werkelijk giftig, maar voorspeld als eetbaar. We voegen het aantal ontbrekende kenmerken en de modelkans toe om patronen in de fouten te onderzoeken.

In [ ]:
error_analysis = X_validation.copy()
error_analysis["werkelijke_klasse"] = y_validation
error_analysis["voorspelde_klasse"] = validation_prediction
error_analysis["kans_giftig"] = validation_probability
error_analysis["ontbrekende_kenmerken"] = X_validation.isna().sum(axis=1)

conditions = [
    (error_analysis["werkelijke_klasse"] == POISONOUS_LABEL)
    & (error_analysis["voorspelde_klasse"] == EDIBLE_LABEL),
    (error_analysis["werkelijke_klasse"] == EDIBLE_LABEL)
    & (error_analysis["voorspelde_klasse"] == POISONOUS_LABEL),
    error_analysis["werkelijke_klasse"] == error_analysis["voorspelde_klasse"],
]
labels = ["false_negative_giftig", "false_positive_giftig", "correct"]
error_analysis["resultaat"] = np.select(conditions, labels, default="onbekend")

display(error_analysis["resultaat"].value_counts().to_frame("aantal"))

plt.figure(figsize=(9, 5))
sns.histplot(
    data=error_analysis,
    x="kans_giftig",
    hue="werkelijke_klasse",
    bins=25,
    stat="density",
    common_norm=False,
    element="step",
)
plt.axvline(FROZEN_THRESHOLD, color="black", linestyle="--", label="drempel 0,46")
plt.title("Voorspelde kans op giftig per werkelijke klasse")
plt.legend(title="Werkelijke klasse", labels=["drempel 0,46", "giftig", "eetbaar"])
plt.show()

most_confident_false_negatives = (
    error_analysis.query("resultaat == 'false_negative_giftig'")
    .sort_values("kans_giftig")
)
display(most_confident_false_negatives.head(15))

## 10. Ontbrekende waarden en categorische foutsegmenten

We vergelijken eerst de missingness per fouttype. Daarna berekenen we per categorie het aantal rijen en het foutenpercentage. Zeer kleine groepen worden niet geïnterpreteerd; de tabel vereist minimaal twintig validatierijen per segment.

In [ ]:
missingness_by_result = (
    error_analysis.groupby("resultaat", observed=True)["ontbrekende_kenmerken"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)
display(missingness_by_result)

error_analysis["is_fout"] = error_analysis["resultaat"] != "correct"
error_analysis["is_false_negative"] = (
    error_analysis["resultaat"] == "false_negative_giftig"
)

segment_rows = []
for feature in categorical_features:
    segment_frame = error_analysis.copy()
    segment_frame[feature] = segment_frame[feature].fillna("__missing__")
    grouped = (
        segment_frame.groupby(feature, observed=True)
        .agg(
            aantal=("is_fout", "size"),
            foutpercentage=("is_fout", "mean"),
            false_negative_percentage=("is_false_negative", "mean"),
        )
        .reset_index()
    )
    grouped = grouped.query("aantal >= 20")
    grouped["feature"] = feature
    grouped = grouped.rename(columns={feature: "waarde"})
    segment_rows.append(grouped)

segment_summary = pd.concat(segment_rows, ignore_index=True)
display(
    segment_summary.sort_values(
        ["false_negative_percentage", "aantal"],
        ascending=[False, False],
    ).head(25).style.format({
        "foutpercentage": "{:.1%}",
        "false_negative_percentage": "{:.1%}",
    })
)

## 11. Modeldisagreement

Als modellen sterk verschillen in hun voorspelde kans, zijn dat interessante gevallen voor handmatige foutanalyse. We berekenen de spreiding tussen de drie basismodellen en tonen de rijen met de grootste onenigheid.

In [ ]:
for model_name, probability in analysis_probabilities.items():
    error_analysis[f"kans_{model_name}"] = probability

probability_columns = [
    f"kans_{model_name}" for model_name in FROZEN_WEIGHTS
]
error_analysis["model_disagreement"] = error_analysis[
    probability_columns
].max(axis=1) - error_analysis[probability_columns].min(axis=1)

disagreement_columns = [
    "werkelijke_klasse",
    "voorspelde_klasse",
    "resultaat",
    "kans_giftig",
    "model_disagreement",
    *probability_columns,
    *categorical_features,
]
display(
    error_analysis.sort_values("model_disagreement", ascending=False)[
        disagreement_columns
    ].head(20)
)

## 12. Interpretatie en beperkingen

Over vijftien gekoppelde folds haalt het bevroren ensemble gemiddeld 0,670 accuracy, 0,848 recall voor giftig en 0,762 F2. De empirische foldintervallen zijn respectievelijk 0,638–0,694, 0,823–0,880 en 0,735–0,789.

Random Forest haalt iets meer recall (0,859) en vrijwel dezelfde F2 (0,764), maar minder accuracy (0,656). Het ensemble verbetert de accuracy in alle vijftien folds met gemiddeld 0,014; de ongecorrigeerde Wilcoxon-p-waarde is 0,0007. Daartegenover daalt recall gemiddeld 0,011. Het F2-verschil van -0,002 is klein en niet overtuigend (p=0,1876). Het ensemble is daarom vooral een betere accuracy/recall-afweging, geen duidelijke F2-doorbraak.

Op de vaste validatieset geeft het ensemble 41 false negatives en 207 false positives. Het gemiddelde aantal ontbrekende kenmerken is vrijwel gelijk bij false negatives en correcte voorspellingen, zodat algemene missingness op zichzelf de fouten niet verklaart. De segment- en disagreementtabellen helpen om specifiekere patronen te onderzoeken.

Belangrijke beperkingen:

- model- en ensemblekeuzes zijn al met dezelfde ontwikkeldata gemaakt, waardoor cross-validatiescores nog enige selectiebias kunnen bevatten;
- herhaalde folds zijn onderling afhankelijk;
- de dataset kan kunstmatige labelruis bevatten;
- correlatie of feature importance bewijst geen oorzaak;
- de testset is nog steeds niet geopend.

CatBoost en stacking zijn niet opnieuw in de vijftien zware folds opgenomen: hun eerdere validatiescores waren niet beter dan de gekozen kandidaten en stacking is computationeel veel duurder. Die beslissing voorkomt veel extra rekentijd zonder de kernvergelijking te verzwakken.

## 13. Testset blijft gesloten

Deze notebook bevat bewust geen code die voorspellingen op `X_test` maakt. De bevroren soft-votingconfiguratie blijft in notebook 05 staan met `RUN_FINAL_TEST = False`.